In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
mu0 = 4 * np.pi * 1e-7

In [3]:
mm = 1e-3

In [20]:
#rectangle coil generator
def generate_rectangular_loop(
    start_point,
    width,
    height,
    plane='xy',
    segments_per_side=100
):
    """
    Generate rectangular loop.

    Parameters
    ----------
    start_point : tuple
        lower-left corner

    width : float
    height : float

    plane : str
        'xy', 'yz', 'xz'

    Returns
    -------
    segment_centers
    dl_vectors
    """
    x0, y0, z0 = start_point

    segment_centers = []
    dl_vectors = []

    # Helper depending on plane

    if plane == 'xy':

        # Left
        ys = np.linspace(y0, y0 - height, segments_per_side, endpoint=False)
        dy = -height / segments_per_side
        
        for y in ys:
            segment_centers.append([x0, y + dy/2, z0])
            dl_vectors.append([0, -dy, 0])

        # Bottom
        xs = np.linspace(x0, x0 + width, segments_per_side, endpoint=False)
        dx = width / segments_per_side

        for x in xs:
            segment_centers.append([x + dx/2, y0 - height, z0])
            dl_vectors.append([dx, 0, 0])

        # Right
        ys = np.linspace(y0 - height, y0, segments_per_side, endpoint=False)
        

        for y in ys:
            segment_centers.append([x0 + width, y - dy/2, z0])
            dl_vectors.append([0, dy, 0])

        # Top
        xs = np.linspace(x0 + width, x0, segments_per_side, endpoint=False)

        for x in xs:
            segment_centers.append([x - dx/2, y0, z0])
            dl_vectors.append([-dx, 0, 0])

        

    elif plane == 'yz':

        ys = np.linspace(y0, y0 + width, segments_per_side, endpoint=False)
        dy = width / segments_per_side

        for y in ys:
            segment_centers.append([x0, y + dy/2, z0])
            dl_vectors.append([0, dy, 0])

        zs = np.linspace(z0, z0 + height, segments_per_side, endpoint=False)
        dz = height / segments_per_side

        for z in zs:
            segment_centers.append([x0, y0 + width, z + dz/2])
            dl_vectors.append([0, 0, dz])

        ys = np.linspace(y0 + width, y0, segments_per_side, endpoint=False)

        for y in ys:
            segment_centers.append([x0, y - dy/2, z0 + height])
            dl_vectors.append([0, -dy, 0])

        zs = np.linspace(z0 + height, z0, segments_per_side, endpoint=False)

        for z in zs:
            segment_centers.append([x0, y0, z - dz/2])
            dl_vectors.append([0, 0, -dz])

    elif plane == 'xz':

        xs = np.linspace(x0, x0 + width, segments_per_side, endpoint=False)
        dx = width / segments_per_side

        for x in xs:
            segment_centers.append([x + dx/2, y0, z0])
            dl_vectors.append([dx, 0, 0])

        zs = np.linspace(z0, z0 + height, segments_per_side, endpoint=False)
        dz = height / segments_per_side

        for z in zs:
            segment_centers.append([x0 + width, y0, z + dz/2])
            dl_vectors.append([0, 0, dz])

        xs = np.linspace(x0 + width, x0, segments_per_side, endpoint=False)

        for x in xs:
            segment_centers.append([x - dx/2, y0, z0 + height])
            dl_vectors.append([-dx, 0, 0])

        zs = np.linspace(z0 + height, z0, segments_per_side, endpoint=False)

        for z in zs:
            segment_centers.append([x0, y0, z - dz/2])
            dl_vectors.append([0, 0, -dz])

    return np.array(segment_centers), np.array(dl_vectors)

In [21]:
# define the coils
segments = 150

coils = {}

In [22]:
#X coils
x_width = 3532 * mm
x_height = 2025 * mm

coils['X1'] = generate_rectangular_loop(
    start_point=(-1700*mm, -1598*mm, -1004.5*mm),
    width=x_width,
    height=x_height,
    plane='yz',
    segments_per_side=segments
)

coils['X2'] = generate_rectangular_loop(
    start_point=(0*mm, -1598*mm, -1004.5*mm),
    width=x_width,
    height=x_height,
    plane='yz',
    segments_per_side=segments
)

coils['X3'] = generate_rectangular_loop(
    start_point=(1700*mm, -1598*mm, -1004.5*mm),
    width=x_width,
    height=x_height,
    plane='yz',
    segments_per_side=segments
)

In [23]:
#Y coils
y_width = 3532 * mm
y_height = 1975 * mm

coils['Y1'] = generate_rectangular_loop(
    start_point=(-1766*mm, 648*mm, -979.5*mm),
    width=y_width,
    height=y_height,
    plane='xz',
    segments_per_side=segments
)

coils['Y2'] = generate_rectangular_loop(
    start_point=(-1766*mm, -1482*mm, -979.5*mm),
    width=y_width,
    height=y_height,
    plane='xz',
    segments_per_side=segments
)

In [24]:
# Zcoils
z_width = 3582 * mm
z_height = 3582 * mm

coils['Z1'] = generate_rectangular_loop(
    start_point=(-1791*mm, 1959*mm, 750*mm),
    width=z_width,
    height=z_height,
    plane='xy',
    segments_per_side=segments
)

coils['Z2'] = generate_rectangular_loop(
    start_point=(-1791*mm, 1959*mm, -750*mm),
    width=z_width,
    height=z_height,
    plane='xy',
    segments_per_side=segments
)

In [25]:
#PMT positions
PMTs = {
    'PMT1': np.array([-760, -834, 0]) * mm,
    'PMT2': np.array([-760, 0, 0]) * mm,
    'PMT3': np.array([-530, 1173, 0]) * mm,
    'PMT4': np.array([530, -837, 0]) * mm,
    'PMT5': np.array([530, 3, 0]) * mm,
    'PMT6': np.array([530, 1173, 0]) * mm,
}

In [26]:
def compute_B_field(point, segment_centers, dl_vectors, current=1.0):
    """
    Compute magnetic field at one point.

    Parameters:
        point : (3,) array

    Returns:
     _total = np.zeros(3)
    """
    B_total = np.zeros(3)
    for r0, dl in zip(segment_centers, dl_vectors):

        r_vec = point - r0
        r_mag = np.linalg.norm(r_vec)

        if r_mag < 1e-12:
            continue

        dB = (
            mu0 * current / (4 * np.pi)
            * np.cross(dl, r_vec)
            / r_mag**3
        )

        B_total += dB

    return B_total

In [27]:
#superposition function

def total_field_at_point(point, coils, currents):

    B_total = np.zeros(3)

    for coil_name in coils:

        segment_centers, dl_vectors = coils[coil_name]

        I = currents[coil_name]

        B = compute_B_field(
            point,
            segment_centers,
            dl_vectors,
            current=I
        )

        B_total += B

    return B_total

In [28]:
currents = {
    'X1':0,
    'X2':0,
    'X3':0,
    'Y1':0,
    'Y2':0,
    'Z1':1,
    'Z2':0,
}

In [29]:
#field at PMTs
results = {}

for pmt_name, position in PMTs.items():

    B = total_field_at_point(position, coils, currents)

    Bmag = np.linalg.norm(B)

    results[pmt_name] = {
        'Bx': B[0],
        'By': B[1],
        'Bz': B[2],
        '|B|': Bmag
    }

In [30]:
T_to_mG = 1e7

for pmt, vals in results.items():

    print(f'{pmt}')
    print('-'*30)

    print(f"Bx   = {vals['Bx'] * T_to_mG:.3f} mG")
    print(f"By   = {vals['By'] * T_to_mG:.3f} mG")
    print(f"Bz   = {vals['Bz'] * T_to_mG:.3f} mG")
    print(f"|B|  = {vals['|B|'] * T_to_mG:.3f} mG")

PMT1
------------------------------
Bx   = -0.556 mG
By   = 0.928 mG
Bz   = 0.129 mG
|B|  = 1.089 mG
PMT2
------------------------------
Bx   = -0.631 mG
By   = 0.102 mG
Bz   = -0.223 mG
|B|  = 0.677 mG
PMT3
------------------------------
Bx   = -0.339 mG
By   = -0.965 mG
Bz   = 0.241 mG
|B|  = 1.051 mG
PMT4
------------------------------
Bx   = 0.339 mG
By   = 0.965 mG
Bz   = 0.241 mG
|B|  = 1.051 mG
PMT5
------------------------------
Bx   = 0.389 mG
By   = 0.105 mG
Bz   = -0.110 mG
|B|  = 0.417 mG
PMT6
------------------------------
Bx   = 0.339 mG
By   = -0.965 mG
Bz   = 0.241 mG
|B|  = 1.051 mG


In [33]:
import numpy as np

currents = {
    'X1':0,
    'X2':0,
    'X3':0,
    'Y1':1,
    'Y2':0,
    'Z1':0,
    'Z2':0,
}

# Z1 center point (meters)

center = np.array([-1.700, 0.168, 0.008])

B = total_field_at_point(center, coils, currents)

print("Field at Z1 center:")
print("Bx =", B[0], "T")
print("By =", B[1], "T")
print("Bz =", B[2], "T")

Field at Z1 center:
Bx = 0.0 T
By = -4.6409106141271263e-07 T
Bz = 0.0 T
